In [ ]:
import json
import re
import pandas as pd


def clean_math_string(s: str) -> str:
    """Normalizes math outputs: safely unwraps \boxed{}, strips $, cleans commas, and parses sentences."""
    if not isinstance(s, str):
        return str(s)

    s = s.strip()

    # Remove wrapping #### or spaces
    s = re.sub(r"^#+", "", s)
    s = re.sub(r"#+$", "", s)
    s = s.strip()

    # Safely extract from \boxed{...} handling nested brackets
    if "\\boxed{" in s:
        start = s.find("\\boxed{") + 7
        depth = 1
        for i in range(start, len(s)):
            if s[i] == "{":
                depth += 1
            elif s[i] == "}":
                depth -= 1
            if depth == 0:
                s = s[start:i]
                break

    s = s.strip("$ \n\t.")

    # Remove conversational prefixes
    s = re.sub(r"^(The )?(final )?answer is\s*:?\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"^Therefore,?\s*(the )?.*?\s+is\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"^So,?\s*(the )?.*?\s+is\s*", "", s, flags=re.IGNORECASE)

    s = s.strip("$ \n\t.")

    # If it is a sentence containing the answer (e.g. "The theater sold 93 tickets")
    # Only triggers if there are no math operators in the sentence.
    if len(s) > 10 and not any(
        op in s for op in ["+", "-", "*", "/", "\\", "^", "=", "x", "y", "z"]
    ):
        nums = re.findall(r"-?[\d,]+(?:\.\d+)?", s)
        if nums:
            s = nums[-1]

    # Remove commas in numbers for comparison (e.g., 6,318 -> 6318)
    if re.match(r"^-?[\d,]+(?:\.\d+)?$", s):
        s = s.replace(",", "")

    return s.strip()


def extract_model_answer(text: str) -> str:
    """Extracts the final answer, handling both strict LoRA formatting and base model fallbacks."""
    ans = "Extraction Failed"

    # 1. Primary Check: The strict MoE format
    if "####" in text:
        parts = text.split("####")
        ans = parts[1] if len(parts) > 1 and parts[1].strip() else parts[-1]

    # 2. Base Model Fallback 1: Standard Instruct phrasing
    elif "The final answer is:" in text:
        ans = text.split("The final answer is:")[-1]

    # 3. Base Model Fallback 2: "The answer is" phrasing
    elif "The answer is:" in text:
        ans = text.split("The answer is:")[-1]

    # 4. Base Model Fallback 3: Conversational ending
    else:
        numbers = re.findall(r"-?[\d,]+(?:\.\d+)?", text)
        if numbers:
            ans = numbers[-1]

    if ans == "Extraction Failed":
        return ans

    return clean_math_string(ans)


In [ ]:
def test_extraction_on_json(json_path: str, max_id: int = -1):
    print(f"Loading results from {json_path}...\n")

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Filter the dataset up until the specified ID
    if max_id != -1:
        data = [item for item in data if item.get("id", 0) <= max_id]

    total = len(data)
    if total == 0:
        print("No samples found matching the criteria.")
        return

    new_correct = 0
    failed_extractions = 0

    print("=" * 80)
    print(
        f" EXTRACTION AUDIT LOG (Processing up to ID: {max_id if max_id != -1 else 'ALL'})"
    )
    print("=" * 80)

    for item in data:
        q_id = item.get("id", "Unknown")
        raw_output = item.get("raw_output", "")

        # Ground truth is passed through the cleaner to ensure formatting matches
        raw_gt = item.get("ground_truth", "")
        clean_gt = clean_math_string(raw_gt)

        # Run our new extraction logic
        new_extracted = extract_model_answer(raw_output)

        is_correct = new_extracted == clean_gt

        if is_correct:
            new_correct += 1
        elif new_extracted == "Extraction Failed":
            failed_extractions += 1

        # We only print the ones that were problematic (either old logic failed, or new logic fails)
        if not item.get("is_correct", False) or not is_correct:
            status = "✅ NEW PASS" if is_correct else "❌ STILL FAILING"
            print(f"\n[ID: {q_id}] - {status}")
            print(f"Old Extracted : {item.get('predicted_answer', '')}")
            print(f"New Extracted : {new_extracted}")
            print(f"Ground Truth  : {clean_gt}")
            if not is_correct:
                print(f"--- Raw Output Tail ---")
                print(raw_output[-150:].strip())  # Print just the end of the generation
            print("-" * 40)

    accuracy = (new_correct / total) * 100
    old_accuracy = (sum(1 for d in data if d.get("is_correct", False)) / total) * 100

    print("\n" + "=" * 50)
    print("FINAL EXTRACTION BENCHMARK")
    print("=" * 50)
    print(f"Total Samples:      {total}")
    print(f"Old Accuracy:       {old_accuracy:.2f}%")
    print(f"New Accuracy:       {accuracy:.2f}%")
    print(f"Failed Extractions: {failed_extractions}")
    print("=" * 50)


# EXECUTE
# Example 1: Run all
# test_extraction_on_json("/content/drive/MyDrive/capstone-project/generated_results/metamath_base_eval.json", max_id=-1)

# Example 2: Run only up to ID 499
test_extraction_on_json("../data/raw/metamath_adapter_eval.json", max_id=100)


# No Instruct Model

In [ ]:
import json
import re
import os


def clean_math_string(s: str) -> str:
    """Normalizes math outputs: safely unwraps \boxed{}, strips $, cleans commas, parses sentences, and truncates loops."""
    if not isinstance(s, str):
        return str(s)

    s = s.strip()

    # 1. Truncate generation collapse loops (newlines, asterisks)
    s = s.split("\n")[0]
    s = s.split("*")[0]

    # 2. Remove non-ASCII spam
    s = re.sub(r"[^\x00-\x7F]+", "", s)

    # Remove wrapping #### or spaces
    s = re.sub(r"^#+", "", s)
    s = re.sub(r"#+$", "", s)
    s = s.strip()

    # Safely extract from \boxed{...} handling nested brackets
    if "\\boxed{" in s:
        start = s.find("\\boxed{") + 7
        depth = 1
        for i in range(start, len(s)):
            if s[i] == "{":
                depth += 1
            elif s[i] == "}":
                depth -= 1
            if depth == 0:
                s = s[start:i]
                break

    s = s.strip("$ \n\t.")

    # Remove conversational prefixes
    s = re.sub(r"^(The )?(final )?answer is\s*:?\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"^Therefore,?\s*(the )?.*?\s+is\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"^So,?\s*(the )?.*?\s+is\s*", "", s, flags=re.IGNORECASE)

    s = s.strip("$ \n\t.")

    # Sentence parsing fallback
    if len(s) > 10 and not any(
        op in s for op in ["+", "-", "*", "/", "\\", "^", "=", "x", "y", "z"]
    ):
        nums = re.findall(r"-?[\d,]+(?:\.\d+)?", s)
        if nums:
            s = nums[-1]

    # Remove commas
    if re.match(r"^-?[\d,]+(?:\.\d+)?$", s):
        s = s.replace(",", "")

    return s.strip()


def extract_model_answer(text: str) -> str:
    ans = "Extraction Failed"

    if "####" in text:
        parts = text.split("####")
        ans = parts[1] if len(parts) > 1 and parts[1].strip() else parts[-1]
    elif "The final answer is:" in text:
        ans = text.split("The final answer is:")[-1]
    elif "The answer is:" in text:
        ans = text.split("The answer is:")[-1]
    else:
        numbers = re.findall(r"-?[\d,]+(?:\.\d+)?", text)
        if numbers:
            ans = numbers[-1]

    if ans == "Extraction Failed":
        return ans

    return clean_math_string(ans)


def re_extract_and_save_json(
    input_json_path: str, output_json_path: str, max_id: int = -1
):
    print(f"Loading raw results from {input_json_path}...\n")

    with open(input_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if max_id != -1:
        data = [item for item in data if item.get("id", 0) <= max_id]

    total = len(data)
    if total == 0:
        print("No samples found.")
        return

    new_correct = 0
    failed_extractions = 0
    output_data = []

    for item in data:
        q_id = item.get("id", "Unknown")
        raw_output = item.get("raw_output", "")
        raw_gt = item.get("ground_truth", "")

        # Process the strings
        clean_gt = clean_math_string(raw_gt)
        new_extracted = extract_model_answer(raw_output)

        is_correct = new_extracted == clean_gt

        if is_correct:
            new_correct += 1
        elif new_extracted == "Extraction Failed":
            failed_extractions += 1

        # Only save the essential fields to the new JSON
        output_data.append(
            {
                "id": q_id,
                "ground_truth": clean_gt,
                "new_extracted": new_extracted,
                "is_correct": is_correct,
            }
        )

    # Calculate Metrics
    accuracy = (new_correct / total) * 100
    old_accuracy = (sum(1 for d in data if d.get("is_correct", False)) / total) * 100

    print("=" * 50)
    print("FINAL EXTRACTION BENCHMARK")
    print("=" * 50)
    print(f"Total Samples:      {total}")
    print(f"Old Accuracy:       {old_accuracy:.2f}%")
    print(f"New Accuracy:       {accuracy:.2f}%")
    print(f"Failed Extractions: {failed_extractions}")
    print("=" * 50)

    # Save to new JSON
    os.makedirs(os.path.dirname(output_json_path), exist_ok=True)
    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4)

    print(f"\n✅ Cleaned extraction data saved successfully to:\n{output_json_path}")


# --- EXECUTE ---
# Update these paths to match your Drive
input_file = "../data/raw/metamath_base_no_instruct_eval.json"
output_file = "../data/results/metamath_base_no_instruct_eval_test.json"

re_extract_and_save_json(input_file, output_file, max_id=-1)
